# 9.4 策略二：GS1 合轴 + 切 S2 负载均衡

9.3 的贪心已经把负载压到行粒度的极限，但**行粒度本身成了新的锯齿**：一行权重 32 块就是 32 块，劈不开。合轴方案把切分粒度推到极限。

## 1. 什么是合轴

```
不合轴（嵌套）:                合轴（展平）:
for b:                          for gs1 in range(B*N*S1):
    for n:                          b, n, s1 = 解码(gs1)
        for s1:                     装箱(gs1 的有效块数)
            装箱(...)
```

**GS1 合轴 = 把 B×N×S1 三层循环展平成一维迭代轴**。`gs1 = b*(N*S1) + n*S1 + s1` 一维单调递增，分核边界可以落在**任意 gs1 位置**——不再受 (b, n) 边界和整行粒度的约束。

## 2. 合轴带来的三个变化

| 维度 | 不合轴 | 合轴 |
|--|--|--|
| 封核边界 | 只能落在行边界 | **任意 gs1 边界**（行内也能切） |
| 切分粒度 | 一行（若干 S2 块） | 一个 gs1 段（可小于一行） |
| 均衡上限 | 行粒度的锯齿 | 基本块粒度的锯齿（**理论极限**） |

合轴后每核任务描述从「(b,n,s1) 区间」变成「**gs1 区间 [start, end)**」——一个 `bnStartIdx` + `gS1StartIdx` 数组即可描述（素材里 `ComputeSplitNBSeq` 的输出正是这两个数组，说明该实现走的就是合轴路线）。

## 3. 代价：kernel 侧的解码成本

合轴把复杂度从 Host 转移到 kernel：

- kernel 拿到 `gs1 ∈ [start, end)`，要**现场解码**出 (b, n, s1) 才能定位数据——解码是几条标量除法/取模（呼应 [8.6](../08_performance_advanced/08.06_scalar_opt.ipynb)：这类计算可以预计算成表放进 TilingData）；
- 跨 (b, n) 边界的核（一段 gs1 横跨两个 head）**内部要切换数据布局**，kernel 代码分「单 BN 核」和「跨 BN 核」两条路径；
- S1 行被劈开时（一个 gs1 段只覆盖半行），softmax 的行统计也要跨核拼接——与切 S2 的 merge 同源。

## 4. 适用场景

| 场景 | 推荐 | 原因 |
|--|--|--|
| B×N 大、S1 块多（prefill/训练） | **不合轴** | 行数远超核数，行粒度锯齿被摊薄，不合轴实现简单 |
| B×N×S1 行数 < 核数，或行权重方差大 | **合轴** | 并行度/粒度是主要矛盾，值得付解码成本 |
| 行数略超核数且长短不齐 | 合轴 | 贪心在行边界留缝，合轴把缝填上 |

## 动手实验：合轴 vs 不合轴的均衡对比


In [ ]:
# 动手：合轴 vs 不合轴在困难场景下的均衡对比
import numpy as np

n_cores = 8
rng = np.random.default_rng(7)

# 困难场景: 行数不多但权重大小悬殊(带 section 的真实形态)
n_rows = 12
weights = rng.integers(8, 120, n_rows)       # 每行有效 S2 块数
total, target = weights.sum(), weights.sum() / n_cores
print(f"行数 {n_rows} < 核数 {n_cores} 的 {n_cores} 核, 总块数 {total}, 期望每核 {target:.1f}")
print(f"行权重: {list(weights)}\n")

# ---- 不合轴: 只能在行边界封核 ----
def split_rows(weights, n_cores):
    cores, cur, cur_w = [], [], 0
    for i, w in enumerate(weights):
        dif = target * (len(cores) + 1) - cur_w
        if cur and w - dif > dif:
            cores.append(cur); cur, cur_w = [], 0
            dif = target * (len(cores) + 1) - cur_w
        cur.append(i); cur_w += w
    if cur: cores.append(cur)
    return [sum(weights[i] for i in c) for c in cores]

loads_unmerged = split_rows(list(weights), n_cores)

# ---- 合轴: gs1 展平, 封核边界可落在行内（块粒度）----
# 展平成块粒度的 gs1 轴后, 贪心可以按块推进, 负载差距不超过 1 个块
flat = np.repeat(np.arange(n_rows), weights)   # 展平: 每个元素 = 一个有效基本块
total_blocks = len(flat)
base, rem = divmod(total_blocks, n_cores)
loads_merged = [base + (1 if i < rem else 0) for i in range(n_cores)]

for name, loads in [('不合轴(行粒度)', loads_unmerged), ('合轴(块粒度)', loads_merged)]:
    used = len([x for x in loads if x > 0])
    dev = (max(loads) - min(x for x in loads if x > 0)) / target
    print(f"{name}: 用核 {used}, 负载 {loads}")
    print(f"  极差/期望 = {dev:.1%}\n")
print(">>> 合轴把粒度压到基本块级, 均衡逼近理论极限, 但 kernel 要付 gs1 解码成本")


### 观察结论

行数 < 核数、行权重悬殊时，不合轴有核分不到任务（或负载极差），合轴把每核负载拉到基本块粒度。**合轴买到的是均衡上限，付出的是 kernel 侧的解码与跨 BN 处理复杂度**——工程上没有免费午餐。

## 5. 工程细节：合轴的三个坑

1. **解码上提**：gs1 → (b, n, s1) 的除法/取模不要在 kernel 循环里现场做，Host 侧预生成查找表；
2. **跨 BN 核的特殊路径**：一段 gs1 横跨两个 (b,n) 时，输入的 base 地址要切换——kernel 需要「先算完本 BN 剩余、再加载下一 BN」的内部小循环；
3. **与切 S2 的叠加**：合轴 + 切 S2 时装箱单元是 (gs1, s2段) 二维块，merge 通路（9.5）同时按行和按段归并——这是全书的均衡复杂度上限。

## 本节要点

- 合轴 = B×N×S1 展平成一维 gs1 轴，封核边界可落在任意位置；
- 收益：粒度从行级到块级，均衡逼近理论极限；代价：gs1 解码、跨 BN 核的特殊路径；
- 选型：prefill/训练（行多）不合轴，decode/行少且不齐（行少）合轴；
- 解码用查找表上提到 TilingData，跨 BN 路径单独分支处理。

## 小测验

1. 合轴与不合轴在封核边界上的本质区别是什么？
2. 合轴方案 kernel 侧新增哪两类复杂度？
3. 什么场景选合轴、什么场景选不合轴？各给出理由。
4. 为什么说「合轴 + 切 S2」是全书均衡复杂度的上限？

>>> 参考答案见 [answer/09.04_answer.txt](answer/09.04_answer.txt)
